In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_parquet("../data/processed/wednesday_clean.parquet")

X = df.drop(columns=["Label", "is_attack"])   # الخصائص (features)
y = df["is_attack"]                            # الهدف: 0 طبيعي، 1 هجوم
labels = df["Label"]                           # نحتفظ بيه لتحليل الأخطاء

print(X.shape, y.shape)

(584980, 66) (584980,)


In [2]:
X_train, X_test, y_train, y_test, lab_train, lab_test = train_test_split(
    X, y, labels,
    test_size=0.2,
    stratify=labels,
    random_state=42,
)

print("Train:", X_train.shape, "  Test:", X_test.shape)

# نتأكد إن النسب متطابقة
pd.DataFrame({
    "train %": lab_train.value_counts(normalize=True) * 100,
    "test %":  lab_test.value_counts(normalize=True) * 100,
}).round(2)


Train: (467984, 66)   Test: (116996, 66)


,train %,test %
Label,,
BENIGN,66.88,66.88
DoS Hulk,29.55,29.55
DoS GoldenEye,1.76,1.76
DoS slowloris,0.92,0.92
DoS Slowhttptest,0.89,0.89


In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
pred = dummy.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print("Recall (attacks caught):", recall_score(y_test, pred))


Accuracy: 0.6688
Recall (attacks caught): 0.0


In [4]:
import time
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

logreg = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced"),
)

start = time.time()
logreg.fit(X_train, y_train)
print(f"Training time: {time.time() - start:.1f} s")

pred_lr = logreg.predict(X_test)
print(classification_report(y_test, pred_lr, target_names=["BENIGN", "ATTACK"], digits=4))


Training time: 8.5 s
              precision    recall  f1-score   support

      BENIGN     0.9896    0.9755    0.9825     78247
      ATTACK     0.9520    0.9793    0.9654     38749

    accuracy                         0.9768    116996
   macro avg     0.9708    0.9774    0.9740    116996
weighted avg     0.9771    0.9768    0.9768    116996



In [5]:
per_type = (
    pd.DataFrame({"label": lab_test.values, "correct": pred_lr == y_test.values})
    .groupby("label")["correct"]
    .agg(["count", "mean"])
    .rename(columns={"count": "n_test", "mean": "detection_rate"})
)
per_type["detection_rate"] = (per_type["detection_rate"] * 100).round(2)
per_type


,n_test,detection_rate
label,,
BENIGN,78247,97.55
DoS GoldenEye,2057,94.75
DoS Hulk,34569,98.64
DoS Slowhttptest,1046,88.34
DoS slowloris,1077,90.44
